## **Introduction to Integer Programming and Applications with Julia**

<table>
  <tr>
    <td>Chapter</td>
    <td>4 - Traveling Salesman Problem</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>4.3 - TSP with no subtour elimination constraints</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [7]:
using JuMP  # Modeling language
using HiGHS # Solver
using HDF5  # For reading HDF5 files

# Load utility functions for plotting
include("utils/tsp_utils.jl")

# Load the distance matrix from the HDF5 file
D = h5read("data/tsp_example.h5", "distance_matrix")

# Number of locations
n = size(D, 1)

# Create the model
model = JuMP.Model(HiGHS.Optimizer)
JuMP.set_silent(model)

# Define the decision variables
@variable(model, x[1:n, 1:n], Bin)

# Objective function: minimize total distance
@objective(model, Min, sum(D[i, j] * x[i, j] for i in 1:n, j in 1:n))

# Constraints: no self-loops
@constraint(model, [i=1:n], x[i, i] == 0)                 

# Constraints: each location visited once
@constraint(model, [j=1:n], sum(x[i, j] for i in 1:n) == 1)

# Constraints: each location departed once
@constraint(model, [i=1:n], sum(x[i, j] for j in 1:n) == 1)

# Solve
JuMP.optimize!(model)

# Get the values of the decision variables
x_opt = JuMP.value.(x)

# Extract the routes from the solution
visited = fill(false, n)
routes = Vector{Vector{Int}}()
for start in 1:n
    if !visited[start]
        route = Int[start]
        visited[start] = true
        nxt = findfirst(x_opt[start, :] .> 0.5)
        while nxt !== nothing && !visited[nxt]
            push!(route, nxt)
            visited[nxt] = true
            nxt = findfirst(x_opt[nxt, :] .> 0.5)
        end
        push!(routes, route)
        println("Route: ", route)
    end
end

# Get the optimal value of the objective function
z_opt = JuMP.objective_value(model)
println("Optimal value: $z_opt meters")

# Plot the solution on the map
fmap = plot_tsp_solution("data/tsp_example.h5", x_opt)
display(fmap)

Route: [1, 2, 4]
Route: [3, 5]
Optimal value: 3040.2 meters


Python: <folium.folium.Map object at 0x7f5c87d79d90>

<table>
  <tr>
    <td>Chapter</td>
    <td>4 - Traveling Salesman Problem</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>4.4 - TSP with MTZ constraints</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [ ]:
using JuMP  # Modeling language
using HiGHS # Solver
using HDF5  # For reading HDF5 files

# Load utility functions for plotting
include("utils/tsp_utils.jl")

# Load the distance matrix from the HDF5 file
D = h5read("data/tsp_example.h5", "distance_matrix")

# Number of locations
n = size(D, 1)

# Create model
model = JuMP.Model(HiGHS.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Define the decision variables
@variable(model, x[1:n, 1:n], Bin)

# Define MTZ variables 
@variable(model, 1 <= u[1:n] <= n)

# Objective function: minimize total distance
@objective(model, Min, sum(D[i,j] * x[i,j] for i in 1:n, j in 1:n))

# Avoid self-loops
@constraint(model, [i=1:n], x[i,i] == 0)

# Each column sums to 1 (each location visited once)
@constraint(model, [j=1:n], sum(x[i,j] for i in 1:n) == 1)

# Each row sums to 1 (each location departs once)
@constraint(model, [i=1:n], sum(x[i,j] for j in 1:n) == 1)

# Breaking symmetry by fixing the first city as the starting point
@constraint(model, u[1] == 1)

# MTZ constraints
@constraint(model, [i=2:n, j=2:n; i != j], u[j] >= u[i] + 1 + n * (x[i,j] - 1))

# Run the solver
JuMP.optimize!(model)

# Get the values of the decision variables
x_opt = JuMP.value.(x)

# Extract the routes from the solution
route = [1]
while true
    to = findfirst(x_opt[route[end], :] .> 0.5)
    to == 1 ? break : push!(route, to)
end
println("Route: $route")

# Get the optimal value of the objective function
z_opt = JuMP.objective_value(model)
println("Optimal value: $z_opt meters")

# Plot the solution on the map
fmap = plot_tsp_solution("data/tsp_example.h5", x_opt)
display(fmap)

Route: [1, 2, 3, 5, 4]
Optimal value: 3664.7 meters


Python: <folium.folium.Map object at 0x7fed552cb770>

<table>
  <tr>
    <td>Chapter</td>
    <td>4 - Traveling Salesman Problem</td>
  </tr>
  <tr>
    <td>Section</td>
    <td>4.5 - TSP with lazy constraints</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

In [1]:
using JuMP   # Modeling language
using GLPK   # Solver
using HDF5   # For reading HDF5 files
using Graphs # For graph operations

# Load utility functions for plotting
include("utils/tsp_utils.jl")

# Define lazy constraints callback function
function tsp_lazy_callback(cb_data)
    
    # Get decision variable values
    x_val = [callback_value(cb_data, x[i,j]) for i in 1:n, j in 1:n]

    # Build an undirected graph
    g = Graphs.SimpleGraph(n)
    for i in 1:n, j in 1:n
        if i != j && x_val[i, j] > 0.5
            Graphs.add_edge!(g, i, j)
        end
    end

    # Find connected components
    cc = Graphs.connected_components(g)

    # If there are more than one connected components, we have a subtour
    if length(cc) > 1
        
        # Pick the smallest component (i.e., subtour)
        minTour = sort(cc, by=length)[1]

        # Initialize the left-hand side of the constraint
        subtourLhs = AffExpr()

        # Sum the decision variables corresponding to the subtour edges
        for i in minTour, j in minTour
            if i != j && x_val[i, j] > 0.5
                subtourLhs += x[i, j]
            end
        end

        # Add lazy constraint to eliminate subtour
        constraint = @build_constraint(subtourLhs <= length(minTour) - 1)

        # Print the constraint for debugging
        println("Adding lazy constraint: ", constraint.func, " <= ", constraint.set.upper)

        # Submit the lazy constraint to the model
        JuMP.MOI.submit(model, JuMP.MOI.LazyConstraint(cb_data), constraint)
    end
end

# Load the distance matrix from the HDF5 file
D = h5read("data/tsp_example.h5", "distance_matrix")

# Number of locations
n = size(D, 1)

# Create model
model = JuMP.Model(GLPK.Optimizer)

# Silent mode (solver output is not printed)
JuMP.set_silent(model)

# Define the decision variables
@variable(model, x[1:n, 1:n], Bin)

# Objective function: minimize total distance
@objective(model, Min, sum(D[i,j] * x[i,j] for i in 1:n, j in 1:n))

# Constraints to eliminate self-loops
@constraint(model, [i=1:n], x[i,i] == 0)

# Each column sums to 1 (each location visited once)
@constraint(model, [j=1:n], sum(x[i,j] for i in 1:n) == 1)

# Each row sums to 1 (each location departs once)
@constraint(model, [i=1:n], sum(x[i,j] for j in 1:n) == 1)

# Set the lazy constraint callback function to the model
JuMP.MOI.set(model, JuMP.MOI.LazyConstraintCallback(), tsp_lazy_callback)

# Run the solver
JuMP.optimize!(model)

# Get the values of the decision variables
x_opt = JuMP.value.(x)

# Extract the routes from the solution
route = [1]
while true
    to = findfirst(x_opt[route[end], :] .> 0.5)
    to == 1 ? break : push!(route, to)
end
println("Route: $route")

# Get the optimal value of the objective function
z_opt = JuMP.objective_value(model)
println("Optimal value: $z_opt meters")

# Plot the solution on the map
fmap = plot_tsp_solution("data/tsp_example.h5", x_opt)
display(fmap)

Adding lazy constraint: x[3,5] + x[5,3] <= 1.0
Adding lazy constraint: x[1,2] + x[2,1] <= 1.0
Route: [1, 4, 5, 3, 2]
Optimal value: 3664.7 meters


Python: <folium.folium.Map object at 0x7fd95b9d3770>